# Notebook 04 Part 1: Spark Internals with Practical Configuration

This notebook belongs to:

```text
Phase 1 — PySpark Foundations & Architecture
Topic: Spark Internals
```

In this notebook, we will clearly understand the main Spark components and how we can control some of them in practical PySpark code.

---

## Topics Covered

```text
1. SparkSession
2. SparkContext
3. Driver
4. Executor
5. Task
6. Partition
7. How to control driver settings
8. How to control local executor/worker threads
9. How to check and change partitions
10. How tasks are related to partitions
11. Practical examples
```

## 1. Big Picture: How Spark Works

When we write PySpark code, Spark internally uses several components.

```text
PySpark Code
    ↓
SparkSession
    ↓
SparkContext
    ↓
Driver
    ↓
Executors
    ↓
Tasks
    ↓
Partitions
```

Simple meaning:

| Spark Component | Simple Meaning |
|---|---|
| SparkSession | Main entry point of PySpark |
| SparkContext | Connection to Spark engine |
| Driver | Main controller of the Spark application |
| Executor | Worker process that runs tasks |
| Task | Small unit of work |
| Partition | Small part of the data |

## 2. Can We Change These Components?

We cannot change all Spark components like normal Python variables.

Some components are controlled using Spark configuration.  
Some are created automatically by Spark.

| Component | Can We Change Directly? | How We Control It |
|---|---|---|
| Driver | Partially | Using driver configs like `spark.driver.memory` |
| Executor | Yes in cluster, limited in local mode | Using `master`, executor configs |
| Task | No direct control | Spark creates tasks automatically |
| Partition | Yes | Using `repartition()` and `coalesce()` |

In this notebook, we will see these practically.

## 3. Windows Local Setup Cell

Run this cell before creating `SparkSession`.

This helps avoid the common Windows local PySpark error:

```text
Timed out while waiting for the Python worker to connect back
```

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"


## 4. Create SparkSession

We will create SparkSession using the Windows-safe configuration.

Important:

```text
local[1] = use 1 local worker thread
```

This is stable for our Windows local setup.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark_Internals_Practical_Config") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 5. Check SparkSession

`spark` is our SparkSession object.

It is entry point for python to spark

We will use it to create DataFrames and access SparkContext.

In [3]:
spark

## 6. Access SparkContext

SparkContext is available inside SparkSession.

```text
SparkSession → SparkContext
```

In [4]:
sc = spark.sparkContext

print("Application Name:", sc.appName)
print("Master:", sc.master)
print("Spark Version:", sc.version)
print("Application ID:", sc.applicationId)

Application Name: Spark_Internals_Practical_Config
Master: local[1]
Spark Version: 4.1.2
Application ID: local-1783643329330


## 7. Understanding Driver

The **Driver** is the main controller of the Spark application.

The Driver:

```text
1. Runs the main program
2. Creates SparkSession
3. Builds execution plan
4. Divides work into tasks
5. Sends tasks to executors
6. Collects final result
```

In our local Jupyter setup, the Driver is running on our own laptop.

## 8. Practical: Driver Configuration

We can control some driver settings using `.config()` while creating SparkSession.

Example:

```python
.config("spark.driver.memory", "2g")
```

This means:

```text
Give 2 GB memory to the Spark driver.
```

Important:

```text
Driver memory should be set before SparkSession is created.
```

If SparkSession is already running, stop it first and create again.

### Example Driver Configuration

Below is an example only. Do not run this if your SparkSession is already active unless you stop the current SparkSession first.

```python
spark.stop()

spark = SparkSession.builder \
    .appName("Driver_Memory_Example") \
    .master("local[1]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()
```

For our current classroom setup, the default driver memory is enough.

## 9. Understanding Executor

Executors are worker processes.

Executors:

```text
1. Receive tasks from Driver
2. Process data partitions
3. Store cached data
4. Return results to Driver
```

In a real cluster, executors run on worker machines.

In our local mode, Spark runs executors/worker threads on the same laptop.

## 10. Practical: Controlling Local Worker Threads

In local mode, we control worker threads using `.master()`.

| Master Setting | Meaning |
|---|---|
| `local[1]` | Use 1 worker thread |
| `local[2]` | Use 2 worker threads |
| `local[4]` | Use 4 worker threads |
| `local[*]` | Use all available CPU cores |

Our setup uses:

```python
.master("local[1]")
```

because it is stable on Windows.

### Example: Using More Threads

If your setup is stable, you can try:

```python
.master("local[2]")
```

or:

```python
.master("local[*]")
```

But for our current Windows setup, we will continue with:

```python
.master("local[1]")
```

## 11. Create Sample DataFrame

Now let us create a DataFrame for practical examples.

In [7]:
data = [
    (1, "Amit", "Delhi", 500),
    (2, "Priya", "Mumbai", 700),
    (3, "Rahul", "Delhi", 300),
    (4, "Sneha", "Pune", 900),
    (5, "Neha", "Mumbai", 400),
    (6, "Karan", "Delhi", 1000),
    (7, "Riya", "Pune", 600),
    (8, "Mohit", "Mumbai", 800)
]

columns = ["order_id", "customer_name", "city", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

+--------+-------------+------+------+
|order_id|customer_name|  city|amount|
+--------+-------------+------+------+
|       1|         Amit| Delhi|   500|
|       2|        Priya|Mumbai|   700|
|       3|        Rahul| Delhi|   300|
|       4|        Sneha|  Pune|   900|
|       5|         Neha|Mumbai|   400|
|       6|        Karan| Delhi|  1000|
|       7|         Riya|  Pune|   600|
|       8|        Mohit|Mumbai|   800|
+--------+-------------+------+------+



## 12. Understanding Partition

A partition is a small part of the data.

Spark divides data into partitions so that different parts can be processed in parallel.

Simple meaning:

```text
Large DataFrame
    ↓
Partition 1
Partition 2
Partition 3
Partition 4
```

Each partition can be processed by a task.

## 13. Check Number of Partitions

We can check the number of partitions using:

```python
df.rdd.getNumPartitions()
```

In [8]:
df.rdd.getNumPartitions()

1

## 14. Practical: Change Number of Partitions using `repartition()`

`repartition()` is used to increase or change the number of partitions.

Example:

```python
df2 = df.repartition(4)
```

This creates a new DataFrame with 4 partitions.

In [9]:
df_4_partitions = df.repartition(4)

print("Original partitions:", df.rdd.getNumPartitions())
print("New partitions:", df_4_partitions.rdd.getNumPartitions())

Original partitions: 1
New partitions: 4


## 15. Important: DataFrame Is Immutable

When we use:

```python
df.repartition(4)
```

the original `df` does not change.

A new DataFrame is created.

That is why we store it in a new variable:

```python
df_4_partitions = df.repartition(4)
```

In [ ]:
print("Original df partitions:", df.rdd.getNumPartitions())
print("df_4_partitions partitions:", df_4_partitions.rdd.getNumPartitions())

## 16. Practical: Reduce Partitions using `coalesce()`

`coalesce()` is used to reduce the number of partitions.

Example:

```python
df_one_partition = df_4_partitions.coalesce(1)
```

This reduces partitions to 1.

In [12]:
df_one_partition = df_4_partitions.coalesce(1)

print("Before coalesce:", df_4_partitions.rdd.getNumPartitions())
print("After coalesce:", df_one_partition.rdd.getNumPartitions())

Before coalesce: 4
After coalesce: 1


## 17. `repartition()` vs `coalesce()`

| Function | Use |
|---|---|
| `repartition()` | Increase or fully reshuffle partitions |
| `coalesce()` | Reduce partitions with less data movement |

Simple rule:

```text
Use repartition() when increasing partitions.
Use coalesce() when reducing partitions.
```

We will study this again in detail in Phase 3: Performance Tuning.

## 18. Understanding Task

A task is a small unit of work created by Spark.

We do not manually create tasks.

Spark creates tasks automatically based on:

```text
1. Number of partitions
2. Type of operation
3. Shuffle requirement
```

Simple rule:

```text
Tasks are usually related to partitions.
```

If a DataFrame has 4 partitions, Spark may create around 4 tasks for a stage.

## 19. Practical: Partitions and Actions

Actions like `count()` and `show()` trigger Spark execution.

When we run an action, Spark creates jobs, stages, and tasks internally.

Let us run `count()`.

In [13]:
df_4_partitions.count() # number of rows

8

## 20. Can We Directly Change Task Count?

Not directly.

Tasks are created by Spark based on data partitions and operations.

But we can indirectly influence task count by changing partitions.

Example:

```python
df.repartition(4)
```

This may create more tasks than:

```python
df.coalesce(1)
```

## 21. Practical Summary Table

| Component | What It Means | How to Control |
|---|---|---|
| Driver | Main controller | Driver configs like `spark.driver.memory` |
| Executor | Worker process | Local mode: `local[1]`, `local[2]`, `local[*]` |
| Task | Unit of work | Spark creates automatically |
| Partition | Small part of data | `repartition()`, `coalesce()` |

## 22. Current Setup Summary

Our current SparkSession:

```python
spark = SparkSession.builder \
    .appName("Spark_Internals_Practical_Config") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()
```

Meaning:

| Code | Meaning |
|---|---|
| `.appName()` | Name of Spark application |
| `.master("local[1]")` | Run locally with 1 worker thread |
| `spark.python.worker.reuse = false` | Avoid Python worker reuse issue |
| `spark.driver.host = 127.0.0.1` | Use localhost |
| `spark.driver.bindAddress = 127.0.0.1` | Bind driver to localhost |

## Stop SparkSession

At the end of the notebook, stop SparkSession to release resources.

In [ ]:
spark.stop()